# 5. Restarting infrastructure-failed jobs in a `ReactionDataset`

This notebook walks through the proper way to restart QCFractal jobs that
ended in `error` for **infrastructure** reasons (segfaults, OOM kills,
I/O errors from full swap, walltime timeouts, transient queue failures) —
as opposed to **chemistry** failures (SCF non-convergence, basis-set
issues, malformed geometry).

## The trap

A BEEP `ReactionDataset` stores **services** (`ReactionRecord`s). Each
reaction has child `SinglepointRecord`s — one per fragment in the
stoichiometry. The reaction's status is **aggregated** from its children,
but **resetting a child does NOT auto-reset its parent service.**

If you reset only the children, the parent stays at `error` and BEEP's
pollers (`check_jobs_status` in `be_hess`, etc.) keep treating it as
failed even after the children compute successfully.

## Why we loop over all four stoichiometries

BEEP splits stoichiometry types into four datasets per cluster:
`{base}_bsse`, `{base}_be_nocp`, `{base}_ie`, `{base}_de`.

- **Child singlepoints** are deduplicated by `(molecule_id, spec)`. A
  shared real-fragment singlepoint that appears in multiple stoichs is
  one record on the server — reset it once, every parent service that
  references it sees the new state.
- **But BSSE has counterpoise (ghost-fragment) singlepoints that don't
  appear in the other three.** Walking only one stoichiometry would miss
  them.
- **Parent `ReactionRecord`s are NOT shared.** Each stoich dataset has
  its own service record per entry — four distinct parent IDs.
  `reset_records` on one stoich's parent does nothing for the others.

The pattern below collects children + parents across all four stoich
datasets, dedupes the child set, and resets in one go.

## 1. Connect

In [ ]:
from qcportal import PortalClient

client = PortalClient("http://localhost:7777", verify=False)
client.server_info["version"]

## 2. Pick the BE base name

Set `BASE` to the BEEP base name (without the `_<stoich>` suffix). The
code below loads all four stoichiometry datasets that exist on the
server.

In [ ]:
BASE = "be_CH2CHCF3_W52_04_HF3C_MINIX"
STOICH_TYPES = ("bsse", "be_nocp", "ie", "de")

datasets = {}
for stoich in STOICH_TYPES:
    name = f"{BASE}_{stoich}"
    try:
        datasets[stoich] = client.get_dataset("reaction", name)
    except Exception as e:
        print(f"  {name}: not found ({e})")

for stoich, ds in datasets.items():
    print(f"{stoich:<8} {ds.name}  entries={len(list(ds.entry_names))}")

## 3. Survey reaction-level status across all stoichs

This is exactly what BEEP's `check_jobs_status` polls — the parent
`ReactionRecord.status`.

In [ ]:
from collections import Counter
from qcportal.record_models import RecordStatusEnum

errored_reactions = []   # list of (stoich, entry_name, spec_name, ReactionRecord)
counts_per_stoich = {}

for stoich, ds in datasets.items():
    counts = Counter()
    for entry_name, spec_name, rec in ds.iterate_records():
        counts[rec.status.value] += 1
        if rec.status == RecordStatusEnum.error:
            errored_reactions.append((stoich, entry_name, spec_name, rec))
    counts_per_stoich[stoich] = dict(counts)

print("Reaction-level status counts per stoichiometry:")
for stoich, c in counts_per_stoich.items():
    print(f"  {stoich:<8} {c}")
print(f"\nTotal errored reaction records across all stoichs: {len(errored_reactions)}")

## 4. Drill into the errored children

`record.children_errors` is a server-side endpoint that returns only the
errored child records — much faster than fetching every component and
filtering client-side. The error message lives on each singlepoint
(`record.error` is a dict with `error_type` + `error_message`).

In [ ]:
# Eyeball the first few errored services to understand failure modes
for stoich, entry_name, spec_name, rxn in errored_reactions[:5]:
    bad_children = rxn.children_errors
    print(f"\n[{stoich}] {entry_name} / {spec_name}  (parent {rxn.id})")
    if not bad_children:
        print("  (parent is error but no child error — stale service state)")
        continue
    for sp in bad_children:
        err = sp.error or {}
        etype = err.get("error_type", "")
        emsg = err.get("error_message", "").splitlines()[0] if err.get("error_message") else ""
        print(f"  child {sp.id}  type={etype!r}\n    {emsg[:200]}")

## 5. Classify: infrastructure vs chemistry

Infrastructure errors are transient and worth retrying as-is. Chemistry
errors are real — the calculation will fail again unless you change the
spec or remove the entry.

The regex list below is a **starting heuristic**. Look at the messages
you actually got in step 4 and extend it for your environment.

In [ ]:
import re

INFRASTRUCTURE_PATTERNS = [
    r"signal 11",                  # SIGSEGV
    r"signal 9",                   # SIGKILL (often OOM)
    r"out of memory",
    r"\bOOM\b",
    r"\bkilled\b",
    r"No space left on device",
    r"Input/output error",
    r"DUE TO TIME LIMIT",          # Slurm walltime
    r"slurm.*time limit",
    r"connection (reset|refused)",
    r"NetworkError",
    r"BrokenPipeError",
    r"executor.*shutdown",
    r"core dumped",
]
INFRA_RE = re.compile("|".join(INFRASTRUCTURE_PATTERNS), re.IGNORECASE)

def is_infrastructure_error(error_blob):
    if not error_blob:
        return False
    text = " ".join(str(v) for v in error_blob.values())
    return bool(INFRA_RE.search(text))

## 6. Build the retry lists

We accumulate across all stoichs and **dedupe child IDs** at the end
(shared singlepoints across stoichs would otherwise appear multiple times).
Three buckets:

- **`child_ids_to_reset`** — singlepoint IDs to push back to `waiting`.
  Deduped across stoichs.
- **`parent_ids_to_reset`** — reaction service IDs to push back. NOT
  deduped — every stoich's parent must be reset independently.
- **`parents_with_real_errors`** — services where at least one child has
  a chemistry-looking error. Don't auto-retry; flag for manual review.

In [ ]:
child_ids_to_reset_set = set()
parent_ids_to_reset = []
parents_with_real_errors = []  # list of (stoich, entry_name, parent_id, [(child_id, error_msg), ...])

for stoich, entry_name, spec_name, rxn in errored_reactions:
    bad_children = rxn.children_errors
    if not bad_children:
        # Stale service state — reset parent only
        parent_ids_to_reset.append(rxn.id)
        continue
    if all(is_infrastructure_error(sp.error) for sp in bad_children):
        child_ids_to_reset_set.update(sp.id for sp in bad_children)
        parent_ids_to_reset.append(rxn.id)
    else:
        details = [
            (sp.id, (sp.error or {}).get("error_message", "")[:120])
            for sp in bad_children
        ]
        parents_with_real_errors.append((stoich, entry_name, rxn.id, details))

child_ids_to_reset = sorted(child_ids_to_reset_set)

print(f"Children to reset (infra errors, deduped): {len(child_ids_to_reset)}")
print(f"Parents to reset (across all stoichs):     {len(parent_ids_to_reset)}")
print(f"Parents skipped (real chemistry):          {len(parents_with_real_errors)}")

## 7. Dry-run inspection

Always look at a sample of what you're about to mutate before committing.
If anything below doesn't look like an infrastructure error, refine the
regex list in step 5 and re-run from step 6.

In [ ]:
print("=== Sample of children flagged as infrastructure error ===")
sample = client.get_records(child_ids_to_reset[:5]) if child_ids_to_reset else []
for sp in sample:
    msg = ((sp.error or {}).get("error_message", "") or "").splitlines()[:1]
    print(f"  {sp.id}  {(msg or [''])[0][:140]}")

print("\n=== Sample of parents skipped due to real errors ===")
for stoich, entry_name, pid, details in parents_with_real_errors[:5]:
    print(f"  [{stoich}] {entry_name} parent={pid}")
    for cid, msg in details:
        print(f"    child {cid}: {msg}")

## 8. Reset

**State-changing.** Children first (they go `error` → `waiting` and the
manager picks them up), then parents (the service re-evaluates and will
transition to `complete` once all children complete).

In [ ]:
if child_ids_to_reset:
    meta = client.reset_records(child_ids_to_reset)
    print(f"Children reset: {meta.n_updated} (of {len(child_ids_to_reset)} requested)")

if parent_ids_to_reset:
    meta = client.reset_records(parent_ids_to_reset)
    print(f"Parents reset:  {meta.n_updated} (of {len(parent_ids_to_reset)} requested)")

## 9. (Optional) Re-tag the retry batch

If you suspect the original compute manager / queue is the source of the
infrastructure failures (e.g. that node has bad memory, or a particular
tag is overloaded), route the retry to a different tag:

```python
client.modify_records(child_ids_to_reset, new_compute_tag="be_retry")
```

Spin up a manager subscribed to that tag on a healthier queue.

## 10. Verify the reset took

In [ ]:
sample_ids = (child_ids_to_reset[:3] + parent_ids_to_reset[:3])
for r in client.get_records(sample_ids):
    print(f"{r.record_type:<10} {r.id:>8}  {r.status.value}")

## 11. Re-run the BEEP workflow

Once the children + parents are back to `waiting`/`running`/`complete`,
the next BEEP run on the same config will:

- **Skip** entries that are now `complete` (cache hit by spec).
- **Wait** for `running`/`waiting` records via the existing `check_jobs_status` polling.

You don't need to delete or re-add anything in the dataset.

## What about the chemistry-error parents?

For `parents_with_real_errors`, the failure is genuine. Options:

- Tighten/loosen the QC spec (different SCF guess, looser convergence).
- Skip the entry: `ds.delete_entries([entry_name])` plus delete the
  reaction record so the next workflow pass doesn't recreate the same
  failure.
- For systematic patterns (one functional always fails on charged species,
  one cluster has a degenerate geometry), fix the input upstream rather
  than reset blindly.